# Visualizing regression results

In [1]:
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../../data/ckpts/rosen'

PARAMS_PATH = '../../3-REGRESSION-MODELS/reports/complete-parameter-estimates.csv'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
fs = grab_all_files(DATA_PATH)
len(fs)

1669

## Importing parameters

In [5]:
dfps = pd.read_csv(PARAMS_PATH)
dfps.head()

,param,beta,std,k,se
0,nx,0.272898,0.055406,4522,0.000824
1,ny,-0.042696,0.035685,4522,0.000531
2,self,-0.217913,1.140945,4522,0.016967


## Processing individual datasets

In [6]:
COEF_VAR = 'I'

In [7]:
# param_list = ['Intercept', 'nx', 'ny', 'self']
param_list = ['nx', 'ny', 'self']

In [8]:
VAL_COL = 'resid'
group_by_cols = ['corpus', 'turn_delta', 'self']

In [9]:
df_scale, df_regression = [], []

In [10]:
for f in tqdm(fs):
    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id', # 'AVAR',
        'nx', 'ny']).to_pandas()

    df = df.loc[
        (df['nx'] >= 5)
        & (df['ny'] >= 5)
    ]

    if ('CANDOR' in f.split('/')[-1]):
        df['corpus'] = 'CANDOR'

    if ('MICASE' in f.split('/')[-1]):
        df['corpus'] = 'MICASE'

    if ('callfriend' in f.split('/')[-1]):
        df['corpus'] = 'callfriend'

    df['self'] = (df['x_speaker'] == df['y_speaker']).astype(int)

    df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df['turn_delta'] = (df['y_turn_id'] - df['x_turn_id'])

    # if ('CANDOR' in f.split('/')[-1]) or ('grace' in f.split('/')[-1]):
    #     df['turn_delta'].loc[df['self'] == 0] -= 1

    if VAL_COL == 'resid':
        df['pred'] = (df['nx'] * dfps['beta'].loc[dfps['param'].isin(['nx'])].values[0]) + (df['ny'] * dfps['beta'].loc[dfps['param'].isin(['ny'])].values[0])

        df['resid'] = df[COEF_VAR] - df['pred']

    df_regression += [
        df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('mean').reset_index()
    ]

    df_regression[-1]['std'] = df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('std').reset_index()[VAL_COL]

    df_regression[-1]['k'] = df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('count').reset_index()[VAL_COL]

    df_regression[-1]['se'] = df_regression[-1]['std'] / np.sqrt(df_regression[-1]['k'])

df_regression = pd.concat(df_regression, ignore_index=True)

  0%|          | 0/1669 [00:00<?, ?it/s]

In [11]:
df0 = df_regression[['turn_delta', 'self'] + [VAL_COL]].groupby(by=['turn_delta', 'self']).agg('mean').reset_index()
df0.head()

,turn_delta,self,resid
0,1,0,-0.443933
1,1,1,-0.942929
2,2,0,-0.299267
3,2,1,-1.970603
4,3,0,-0.879937


In [12]:
df1 = df_regression[['turn_delta', 'self', 'corpus'] + [VAL_COL]].groupby(by=['turn_delta', 'self', 'corpus']).agg('mean').reset_index()
df1.head(n=25)

,turn_delta,self,corpus,resid
0,1,0,CABNC,-0.453505
1,1,0,CANDOR,-0.444693
2,1,0,CORAAL,-0.511283
3,1,0,DISPEL,-0.501284
4,1,0,Frederiksen,-0.403032
5,1,0,GCSAusE,-0.123604
6,1,0,Graesser,0.268663
7,1,0,MICASE,-0.513907
8,1,0,SBCSAE,-0.179193
9,1,0,SCoSE,-0.267159


## Plotly vis

In [13]:
import plotly.graph_objects as go

### Aggregate/total visualization

Some of the corpora have different structures overall. This means that, for say CANDOR and the MIAO corpora, there is a strange distribution of turns such that self is always even turns away and other is always odd turns away. Especially because CANDOR is such a large component of the data, this caused a bumpier distribution than one would have expected.

To solve for this, we averaged odd versus even turns, smoothening out the overall visualization of the distribution.

##### Merge and average values

In [14]:
df01 = df0[['turn_delta', 'self', VAL_COL]].copy()
df01['turn_delta'] -= 1

In [15]:
df0 = pd.merge(
    left=df0, left_on=['turn_delta', 'self'],
    right=df01, right_on=['turn_delta', 'self'],
    how='left'
)
df0.head()

,turn_delta,self,resid_x,resid_y
0,1,0,-0.443636,-0.291997
1,1,1,-0.908549,-1.968931
2,2,0,-0.291997,-0.879161
3,2,1,-1.968931,-0.761422
4,3,0,-0.879161,-0.315018


In [16]:
df0['resid'] = df0[['resid_x', 'resid_y']].mean(axis=1)
df0.head()

,turn_delta,self,resid_x,resid_y,resid
0,1,0,-0.443636,-0.291997,-0.367817
1,1,1,-0.908549,-1.968931,-1.438740
2,2,0,-0.291997,-0.879161,-0.585579
3,2,1,-1.968931,-0.761422,-1.365177
4,3,0,-0.879161,-0.315018,-0.597089


### Main vis

In [17]:
sel_1 = (df0['turn_delta'] <= 200) & (df0['turn_delta'] > 0) & ((df0['turn_delta'] % 2) == 0)

In [18]:
fig = go.Figure()

In [19]:
sel = df0['self'] == 1
fig.add_trace(
    go.Scatter(
        y = df0[VAL_COL].loc[sel & sel_1].values,
        # x = np.array(range(int((sel & sel_1).sum()))) + 1,
        x = df0['turn_delta'].loc[sel & sel_1].values,
        mode='lines',
        name='self',
        showlegend=True,
        legendgroup='self',
        line = dict(
            color='orange'
        )
    )
)

sel = df0['self'] == 0
fig.add_trace(
    go.Scatter(
        y = df0[VAL_COL].loc[sel & sel_1].values,
        # x = np.array(range(int((sel & sel_1).sum()))) + 1,
        x = df0['turn_delta'].loc[sel & sel_1].values,
        mode='lines',
        name='other',
        showlegend=True,
        legendgroup='other',
        line = dict(
            color='royalblue'
        )
    )
)

fig.update_layout(template='xgridoff')

In [20]:
fig.write_html(os.path.join(OUTPUTS_PATH, 'all-corpora.html'))
fig.write_image(os.path.join(OUTPUTS_PATH, 'all-corpora.png'), scale=5)

### Per corpus vis

In [14]:
figs = []

In [15]:
for corpus in df1.corpus.unique():
    sub = df1.loc[df1['corpus'].isin([corpus])]

    fig = go.Figure()

    sel_t = (sub['turn_delta'] > 0) & (sub['turn_delta'] <= 200)

    sel = sub['self'] == 1
    fig.add_trace(
        go.Scatter(
            x=sub['turn_delta'].loc[sel & sel_t].values,
            y = sub[VAL_COL].loc[sel & sel_t].values,
            mode='lines',
            name='self',
            showlegend=True,
            legendgroup='self',
            line = dict(
                color='orange'
            )
        )
    )

    sel = sub['self'] == 0
    fig.add_trace(
        go.Scatter(
            x=sub['turn_delta'].loc[sel & sel_t].values,
            y = sub[VAL_COL].loc[sel & sel_t].values,
            mode='lines',
            name='other',
            showlegend=True,
            legendgroup='other',
            line = dict(
                color='royalblue'
            )
        )
    )

    fig.update_layout(template='xgridoff')

    figs += [fig]

In [16]:
for corpus, plot in zip(df1.corpus.unique(), figs):
    print(corpus)
    plot.show()
    plot.write_html(os.path.join(OUTPUTS_PATH, corpus+'.html'))
    plot.write_image(os.path.join(OUTPUTS_PATH, corpus+'.png'), scale=5)

CABNC


CANDOR


CORAAL


DISPEL


Frederiksen


...

SBCSAE


SCoSE


callfriend


grace


med_school
